In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage, ToolMessage
from rich import print as rprint

load_dotenv(override=True)

api_key = os.getenv("DEEPSEEK_API_KEY")
base_url = os.getenv("DEEPSEEK_API_BASE")

chat_model = init_chat_model(api_key=api_key, base_url=base_url,model='deepseek:deepseek-flash')

### 多轮会话处理

In [15]:
messages = [
    {
        "role": "system",
        "content": "你是一名 AI 助手",
    },
    {
        "role": "user",
        "content": "上海天气如何？",
    },
    {
        "role": "assistant",
        "content": "天气很好",
    },
    {
        "role": "user",
        "content": "你是谁",
    },
    {
        "role": "assistant",
        "content": "我是ai助手",
    },
]


def get_msg(msgs) -> list:
    system_message = [m for m in msgs if m["role"] == "system"]
    chat_message = [m for m in msgs if m["role"] != "system"]

    chat_message = chat_message[-2:]
    return system_message + chat_message


final_msgs = get_msg(messages)
final_msgs.append(
    {
        "role": "user",
        "content": "我问的第一个问题是什么",
    }
)
response = chat_model.invoke(final_msgs)
rprint(response)

AIMessage(
    content='你问的第一个问题是：**“你是谁”**。',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 
'我们需要回答用户中文问题。用户问“我问的第一个问题是什么”。需要回顾对话。对话开始：用户说“你是谁”，我回答“我是ai助
手”。然后用户问“我问的第一个问题是什么”。所以第一个问题是“你是谁”。需要直接回答。注意可能用户是问在当前会话中第一个
问题。答案：你问的第一个问题是“你是谁”。可以简洁。'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 87,
            'prompt_tokens': 47,
            'total_tokens': 134,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 75,
                'rejected_prediction_tokens': None,
                'text_tokens': None
            },
            'prompt_tokens_details': {
                'audio_tokens': None,
                'cache_write_tokens': None,
                'cached_tokens': 0,
                'image_tokens': None,
                'text_tokens': None
            },
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 47
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': '24767995-c3ef-4fba-bf94-21ca9e19e54b',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a0c9b2-d6dc-7252-9a99-f9abe5af50d3-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 47,
        'output_tokens': 87,
        'total_tokens': 134,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 75}
    }
)

###  调用工具tool

In [2]:
from langchain_core.tools import tool

messages: list[BaseMessage] = [
    SystemMessage(content='你是一个ai助手，回答问题尽量使用给你的工具'),
    HumanMessage(content='上海今天多少度')
]


@tool
def weather_tool(city: str):
    """获取指定城市的天气"""
    return f'{city}今天99度'


tools = {weather_tool.name: weather_tool}

# 给大模型绑定可使用的工具
chat_model_with_tools = chat_model.bind_tools([weather_tool])

# 第一次请求大模型，大模型判断是否需要调用工具
assistant_message = chat_model_with_tools.invoke(messages)
messages.append(assistant_message)
rprint(assistant_message)

# 大模型需要调用工具，本地执行工具将toolMessage加入到messages（不适合工具回答之后还需要再调用新工具的情况）
for tool_call in assistant_message.tool_calls:
    tool = tools[tool_call["name"]]
    tool_message = tool.invoke(tool_call)
    rprint(tool_message)
    messages.append(tool_message)

# 根据tools结果再请求一次大模型，汇总结果
final_assistant_message = chat_model_with_tools.invoke(messages)
rprint(final_assistant_message)


AIMessage(
    content='',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': "The user is asking about Shanghai's weather today. I should use the weather tool."
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 56,
            'prompt_tokens': 302,
            'total_tokens': 358,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 17,
                'rejected_prediction_tokens': None,
                'text_tokens': None
            },
            'prompt_tokens_details': {
                'audio_tokens': None,
                'cache_write_tokens': None,
                'cached_tokens': 128,
                'image_tokens': None,
                'text_tokens': None
            },
            'prompt_cache_hit_tokens': 128,
            'prompt_cache_miss_tokens': 174
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': 'dd21bc07-2c5b-4a44-a564-468d3c372026',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--01a0c960-e7a5-7d61-8216-17340e2039b9-0',
    tool_calls=[
        {
            'name': 'weather_tool',
            'args': {'city': '上海'},
            'id': 'call_00_oMccusTrB9P0PmVyCVlY7716',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 302,
        'output_tokens': 56,
        'total_tokens': 358,
        'input_token_details': {'cache_read': 128},
        'output_token_details': {'reasoning': 17}
    }
)

ToolMessage(content='上海今天99度', name='weather_tool', tool_call_id='call_00_oMccusTrB9P0PmVyCVlY7716')

AIMessage(
    content='上海今天的气温是 **99 
度**。\n\n不过要提醒一下，这个数值听起来不太像是正常的气温（无论是摄氏度还是华氏度都偏高得离谱），可能是数据源出了
点问题。建议你再通过其他渠道（比如天气App）核实一下哦。',
    additional_kwargs={'refusal': None, 'reasoning_content': ''},
    response_metadata={
        'token_usage': {
            'completion_tokens': 58,
            'prompt_tokens': 374,
            'total_tokens': 432,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 0,
                'rejected_prediction_tokens': None,
                'text_tokens': None
            },
            'prompt_tokens_details': {
                'audio_tokens': None,
                'cache_write_tokens': None,
                'cached_tokens': 128,
                'image_tokens': None,
                'text_tokens': None
            },
            'prompt_cache_hit_tokens': 128,
            'prompt_cache_miss_tokens': 246
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': 'ffea17ab-29a6-475c-be93-59398ff38691',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a0c960-eb59-7222-9ae4-efac64d6b0c1-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 374,
        'output_tokens': 58,
        'total_tokens': 432,
        'input_token_details': {'cache_read': 128},
        'output_token_details': {'reasoning': 0}
    }
)

### 使用tool,并且在LangSmith中只存在一条调用

In [2]:
from langchain_core.tools import tool
from langsmith import traceable


@tool
def weather_tool(city: str) -> str:
    """获取指定城市的天气。"""
    return f"{city}今天99度"


# 注册可用工具
tools = {
    weather_tool.name: weather_tool,
}

# 将工具描述和参数 Schema 绑定给模型
chat_model_with_tools = chat_model.bind_tools([weather_tool])


# 一次会话的多次请求，在langsmith中中存在一条记录
@traceable(
    name="weather-agent-request",
    run_type="chain",
    tags=["weather", "manual-tool-calling"],
)
def ask_weather(question: str, max_tool_rounds: int = 5) -> str:
    """执行一次完整的天气问答。"""

    # messages 必须放在函数内部，避免不同请求之间相互污染
    messages: list[BaseMessage] = [
        SystemMessage(content="你是一个AI助手，回答问题时尽量使用提供的工具。"),
        HumanMessage(content=question),
    ]

    tool_rounds = 0

    while True:
        # 请求模型，由模型判断是否调用工具
        assistant_message = chat_model_with_tools.invoke(messages)
        messages.append(assistant_message)
        rprint(assistant_message)

        # 没有工具调用，说明模型已经生成最终答案
        if not assistant_message.tool_calls:
            return str(assistant_message.content)

        if tool_rounds >= max_tool_rounds:
            raise RuntimeError(f"工具调用超过最大轮数：{max_tool_rounds}")

        # 执行本轮所有工具调用
        for tool_call in assistant_message.tool_calls:
            tool_name = tool_call["name"]
            selected_tool = tools.get(tool_name)

            if selected_tool is None:
                raise ValueError(f"模型调用了未注册的工具：{tool_name}")

            # 传入完整 ToolCall，LangChain 会返回对应的 ToolMessage
            tool_message = selected_tool.invoke(tool_call)
            messages.append(tool_message)
            rprint(tool_message)

        tool_rounds += 1


answer = ask_weather("上海今天多少度")
rprint(answer)

AIMessage(
    content='',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 'The user is asking about the weather in Shanghai. I should use the weather tool.'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 56,
            'prompt_tokens': 304,
            'total_tokens': 360,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 17,
                'rejected_prediction_tokens': None,
                'text_tokens': None
            },
            'prompt_tokens_details': {
                'audio_tokens': None,
                'cache_write_tokens': None,
                'cached_tokens': 128,
                'image_tokens': None,
                'text_tokens': None
            },
            'prompt_cache_hit_tokens': 128,
            'prompt_cache_miss_tokens': 176
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': '468cda28-09a0-4d33-9b1e-2e9007ae8ff5',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--01a0ccf2-fcc5-7890-8f0a-7df825e8d68d-0',
    tool_calls=[
        {
            'name': 'weather_tool',
            'args': {'city': '上海'},
            'id': 'call_00_HpoJhfryEbd1XvQZJOR05215',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 304,
        'output_tokens': 56,
        'total_tokens': 360,
        'input_token_details': {'cache_read': 128},
        'output_token_details': {'reasoning': 17}
    }
)

ToolMessage(content='上海今天99度', name='weather_tool', tool_call_id='call_00_HpoJhfryEbd1XvQZJOR05215')

AIMessage(
    content='上海今天的气温是 **99 
度**。\n\n不过这个数值没有标明单位——如果是摄氏度（99℃）显然不符合实际，全球有记录以来的最高气温也远低于此；更可能是
华氏度，99°F 约等于 37.2℃，属于炎热天气。建议你确认一下数据来源的单位，或查看更权威的天气预报。',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 'The tool returned "上海今天99度" (Shanghai is 99 degrees today). That seems like an 
odd value. It\'s likely Celsius 99 which is unrealistic, but it might be Fahrenheit? Actually the tool returned it.
I should report it. 99 degrees — presumably the unit is unclear. Let me just report what the tool said, noting the 
unit is likely Fahrenheit (99°F ≈ 37°C) since 99°C would be impossible.\n\nActually, I should be careful about 
over-interpreting. I\'ll report the result and note the likely unit.'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 194,
            'prompt_tokens': 376,
            'total_tokens': 570,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 118,
                'rejected_prediction_tokens': None,
                'text_tokens': None
            },
            'prompt_tokens_details': {
                'audio_tokens': None,
                'cache_write_tokens': None,
                'cached_tokens': 128,
                'image_tokens': None,
                'text_tokens': None
            },
            'prompt_cache_hit_tokens': 128,
            'prompt_cache_miss_tokens': 248
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': 'cd6bcc0d-75da-42e8-92b7-5394c3db78fe',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a0ccf3-00a3-7980-917c-630d32edffc5-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 376,
        'output_tokens': 194,
        'total_tokens': 570,
        'input_token_details': {'cache_read': 128},
        'output_token_details': {'reasoning': 118}
    }
)

上海今天的气温是 **99 度**。

不过这个数值没有标明单位——如果是摄氏度（99℃）显然不符合实际，全球有记录以来的最高气温也远低于此；更可能是华氏度，99
°F 约等于 37.2℃，属于炎热天气。建议你确认一下数据来源的单位，或查看更权威的天气预报。